<a href="https://colab.research.google.com/github/henrypitts/Baseball-Colab-Tools/blob/main/Pitchers_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- 1) SETUP & CONFIG ---
import os, math, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.backends.backend_pdf import PdfPages
from scipy.stats import gaussian_kde

# Path to your master CSV
CSV_PATH = "DBWS GM 1 BUILD.csv"   # <-- update if needed

# Output folder
OUT_DIR = "/content/pitcher_reports"
os.makedirs(OUT_DIR, exist_ok=True)

# General look
plt.rcParams["font.family"] = "DejaVu Sans"
DPI = 200

# Strike zone bounds (feet)
SZ_HALF_W = 0.83
SZ_TOP, SZ_BOT = 3.5, 1.5

# Movement plot fixed range (inches)
MOV_LIMIT = 25

# Toggle graphs
SHOW_VELO_RIDGE = True
SHOW_TILT_ROSE  = True
SHOW_USAGE_PIE  = True
SHOW_LOCATIONS  = True
SHOW_MOVEMENT   = True

# Pitch colors – GWU / Professional Red-Black-Grey-Yellow-Orange palette
COLOR_MAP = {
    # 1. Four-seamers / Hard Fastballs – Red
    "FF": "#C8102E", "FourSeam": "#C8102E", "Fastball": "#C8102E",

    # 2. Sliders – Black
    "SL": "#000000", "Slider": "#000000",

    # 3. Changeups – Light Grey
    "CH": "#BFC0C0", "Changeup": "#BFC0C0",

    # 4. Cutters – Dark Orange
    "FC": "#D35400", "Cutter": "#D35400",

    # 5. Curveballs – Warm Yellow
    "CU": "#F4C430", "Curveball": "#F4C430",

    # 6. Sinkers / Two-Seamers – Dark Grey
    "SI": "#4B4B4B", "Sinker": "#4B4B4B",

    # Splitters – Deep Red highlight for visual pop
    "ST": "#A6192E", "Sweeper": "#A6192E",
    # 8. Knuckleballs – Bright Yellow highlight
    "KN": "#FFD700", "Knuckleball": "#FFD700",
}



# Aliases for flexible column detection
ALIASES = {
    "pitcher": ["Pitcher","PitcherName","Player"],
    "pitch_type": ["TaggedPitchType","PitchType","Type","pitch_type"],
    "velo": ["RelSpeed","Velocity","Velo","PitchVelo","StartSpeed","ReleaseSpeed"],
    "hb": ["HorzBreak","HB","XBreak","HorizontalBreak"],
    "ivb": ["InducedVertBreak","IVB","YBreak","VerticalBreak","VertBreak"],
    "zone_x": ["PlateLocSide","px","PlateX","ZoneX"],
    "zone_z": ["PlateLocHeight","pz","PlateZ","ZoneZ"],
    "pitch_call": ["PitchCall","Call","Result","PitchResult"],
    "play_result": ["PlayResult","BattedBallType","PlayOutcome"],
    "exit_velo": ["ExitSpeed","ExitVelo","EV","BattedBallSpeed"],
    "spin_axis": ["SpinAxis","Observed spin axis (deg)","Axis_Observed","Spin_Axis","Spin Axis"],
}


In [ ]:
# --- 2) HELPERS ---
def col(df, key):
    for name in ALIASES.get(key, []):
        if name in df.columns: return name
    return None

def pretty_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def kde_ridge(ax, values, y, span=None, label=None, color=None):
    vals = np.asarray(values, dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) < 1:
        return
    # KDE if we have >1 point; always draw mean marker
    if len(vals) > 1:
        lo = span[0] if span else np.nanmin(vals)
        hi = span[1] if span else np.nanmax(vals)
        kde = gaussian_kde(vals)
        x = np.linspace(lo, hi, 200)
        z = kde(x); z = z / z.max() * 0.8
        ax.fill_between(x, y - z/2, y + z/2, alpha=0.25, color=color)
        ax.plot(x, np.full_like(x, y), lw=0.8, color=color)
    m = float(np.mean(vals))
    ax.scatter([m], [y], s=40, zorder=3, color=color, edgecolors="white", linewidths=0.5)
    if label:
        ax.text(m, y + 0.12, label, fontsize=8, ha="center", color=color)

def to_polar_theta_deg(clock_deg):
    # Convert "clock style" deg (0 top, cw) -> matplotlib polar (0 east, ccw)
    return (90.0 - clock_deg) % 360.0


In [ ]:
# --- 3) LOAD DATA ---
df_all = pd.read_csv(CSV_PATH)

# Normalize % strings
for c in df_all.columns:
    if df_all[c].dtype == object and df_all[c].astype(str).str.endswith("%").any():
        df_all[c] = pd.to_numeric(df_all[c].astype(str).str.replace("%","",regex=False), errors="coerce")

# Column picks
PITCHER = col(df_all,"pitcher")
PTYPE   = col(df_all,"pitch_type")
VELO    = col(df_all,"velo")
HB      = col(df_all,"hb")
IVB     = col(df_all,"ivb")
ZX      = col(df_all,"zone_x")
ZZ      = col(df_all,"zone_z")
PCALL   = col(df_all,"pitch_call")
PPLAY   = col(df_all,"play_result")
EV      = col(df_all,"exit_velo")
SPINAX  = col(df_all,"spin_axis")

if PITCHER is None or PTYPE is None:
    raise ValueError("Need Pitcher and Pitch Type columns.")

def _norm(s): return str(s).strip().lower() if isinstance(s,str) else s

def summarize_pitcher(sub):
    cols = [PTYPE]
    if VELO: cols.append(VELO)
    if IVB:  cols.append(IVB)
    if HB:   cols.append(HB)
    if EV:   cols.append(EV)
    if PCALL: cols.append(PCALL)
    if PPLAY: cols.append(PPLAY)
    d = sub[cols].copy()

    if PCALL:
        pcn = d[PCALL].map(_norm)
        d["is_strike"] = pcn.isin({"strikecalled","strikeswinging","foul","foultip"})
        d["is_swing"]  = pcn.isin({"strikeswinging","foul","foultip","inplay"})
        d["is_bip"]    = pcn.eq("inplay")
        d["is_whiff"]  = pcn.eq("strikeswinging")
    else:
        d["is_strike"]=d["is_swing"]=d["is_bip"]=d["is_whiff"]=False
    if PPLAY in d.columns:
        prn = d[PPLAY].map(_norm)
        d["is_bip"] = d["is_bip"] | prn.isin({"inplay","inplayout","inplaynoout","inplayrun",
                                              "groundball","flyball","line drive","linedrive",
                                              "popup","pop-up","popout"})

    grp = d.groupby(PTYPE, dropna=False)
    def avg(s): s = pd.to_numeric(s,errors="coerce"); return round(np.nanmean(s),1) if len(s) else np.nan
    def maxv(s): s = pd.to_numeric(s,errors="coerce"); return round(np.nanmax(s),1) if len(s) else np.nan

    table = pd.DataFrame({
        "Count": grp.size(),
        "Avg Velo": grp[VELO].apply(avg) if VELO else np.nan,
        "Max Velo": grp[VELO].apply(maxv) if VELO else np.nan,
        "Avg IVB":  grp[IVB].apply(avg) if IVB else np.nan,
        "Avg HB":   grp[HB].apply(avg) if HB else np.nan,
        "Strikes":  grp["is_strike"].sum(),
        "Swings":   grp["is_swing"].sum(),
        "Whiffs":   grp["is_whiff"].sum(),
        "BIP":      grp["is_bip"].sum(),
        "Avg EV":   grp[EV].apply(avg) if EV else np.nan,
    }).reset_index().rename(columns={PTYPE:"Pitch Type"})
    return table.sort_values("Count", ascending=False)


In [ ]:
# --- 4) REPORT BUILDER ---
def draw_pitcher_page(sub, pitcher_name):
    table = summarize_pitcher(sub)
    types = table["Pitch Type"].tolist()
    fig = plt.figure(figsize=(11,14), dpi=DPI)
    gs = GridSpec(8,6,figure=fig)

    # Title
    ax_title = fig.add_subplot(gs[0,:]); ax_title.axis("off")
    ax_title.text(0.01,0.6,f"{pitcher_name} – Pitch Report",fontsize=20,weight="bold")

    # Summary table
    ax_tbl = fig.add_subplot(gs[1:3,:6]); ax_tbl.axis("off")
    show_cols = ["Pitch Type","Count","Avg Velo","Max Velo","Avg IVB","Avg HB",
                 "Strikes","Swings","Whiffs","BIP","Avg EV"]
    t = ax_tbl.table(cellText=table[show_cols].fillna("").values,
                     colLabels=show_cols,loc="center",cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1,1.25)
    for (r,c),cell in t.get_celld().items():
        if r==0: cell.set_text_props(weight="bold")

    # Velocity ridgeline (colorized + guard for empty velo)
    if SHOW_VELO_RIDGE and VELO in sub.columns:
        ax_vel = fig.add_subplot(gs[3:5,3:6]); pretty_ax(ax_vel)
        all_v = pd.to_numeric(sub[VELO], errors="coerce")
        finite = all_v[np.isfinite(all_v)]
        if len(finite) > 0:
            vspan = (finite.min() - 1.5, finite.max() + 1.5)
            for i, tname in enumerate(types):
                vals = pd.to_numeric(sub.loc[sub[PTYPE]==tname, VELO], errors="coerce")
                c = COLOR_MAP.get(tname, "#555555")
                # If your kde_ridge doesn't accept 'color', remove color= and it will still work;
                # but to get colored ridges, update kde_ridge to include a 'color' param.
                try:
                    kde_ridge(ax_vel, vals, len(types)-i, span=vspan, label=None, color=c)
                except TypeError:
                    # fallback if kde_ridge has no 'color' arg yet
                    kde_ridge(ax_vel, vals, len(types)-i, span=vspan)
                # label
                m = np.nanmean(vals) if np.isfinite(vals).any() else np.mean(vspan)
                ax_vel.text(m, len(types)-i+0.12, f"{tname} (n={vals.notna().sum()})",
                            fontsize=8, ha="center", color=c)
            ax_vel.set_xlabel("Velocity (mph)"); ax_vel.set_yticks([])
            ax_vel.set_title("Velocity Consistency")
        else:
            ax_vel.text(0.5, 0.5, "No velocity data", ha="center", va="center",
                        transform=ax_vel.transAxes)

    # Spin axis / tilt rose
    if SHOW_TILT_ROSE and (SPINAX in sub.columns):
        ax_tilt = fig.add_subplot(gs[3:5,0:3], projection="polar")
        ax_tilt.set_title("Tilt Consistency (Spin Axis)", fontsize=11, pad=25)
        ax_tilt.set_theta_zero_location("N"); ax_tilt.set_theta_direction(-1)
        any_points = False
        for tname in types:
            s = pd.to_numeric(sub.loc[sub[PTYPE]==tname, SPINAX], errors="coerce").dropna()
            if not len(s): continue
            any_points = True
            th = np.deg2rad(to_polar_theta_deg(s.values))
            ax_tilt.scatter(th, np.ones_like(th), s=14, alpha=0.6,
                            c=COLOR_MAP.get(tname,"#555555"), label=tname)
        if any_points:
            ax_tilt.legend(loc="lower center", bbox_to_anchor=(0.5,-0.15),
                           ncol=4, fontsize=8, frameon=False)
        else:
            ax_tilt.text(0.5, 0.5, "No spin-axis data", ha="center", va="center",
                         transform=ax_tilt.transAxes)

    # Usage pie chart (fixed return unpacking + colors)
    if SHOW_USAGE_PIE:
        ax_pie = fig.add_subplot(gs[5:7,0:2])
        counts = sub[PTYPE].value_counts(dropna=False)
        if counts.sum() > 0:
            labels = [str(k) for k in counts.index]
            pie_colors = [COLOR_MAP.get(lbl, "#555555") for lbl in labels]
            wedges, texts, autotexts = ax_pie.pie(
                counts.values,
                labels=labels,
                autopct="%1.0f%%",
                startangle=90,
                textprops={'color': 'white'},
                colors=pie_colors
            )
        else:
            ax_pie.text(0.5, 0.5, "No pitches", ha="center", va="center",
                        transform=ax_pie.transAxes)
        ax_pie.set_title("Pitch Usage")

    # Location scatter
    if SHOW_LOCATIONS and ZX and ZZ:
        ax_loc = fig.add_subplot(gs[5:7,2:4]); pretty_ax(ax_loc)
        # zone box
        zone = plt.Rectangle((-SZ_HALF_W, SZ_BOT), 2*SZ_HALF_W, SZ_TOP-SZ_BOT,
                             fill=False, lw=1.2, color="#333")
        ax_loc.add_patch(zone)
        ax_loc.set_xlim(-2, 2); ax_loc.set_ylim(0, 5)
        plotted_any = False
        for tname in types:
            dd = sub.loc[sub[PTYPE]==tname, [ZX, ZZ]].copy()
            dd[ZX] = pd.to_numeric(dd[ZX], errors="coerce")
            dd[ZZ] = pd.to_numeric(dd[ZZ], errors="coerce")
            dd = dd.dropna()
            if len(dd):
                plotted_any = True
                ax_loc.scatter(dd[ZX], dd[ZZ], s=16, alpha=0.5, label=tname,
                               c=COLOR_MAP.get(tname,"#555555"))
        if plotted_any:
            ax_loc.legend(fontsize=7, frameon=False)
        else:
            ax_loc.text(0.5, 0.5, "No location data", ha="center", va="center",
                        transform=ax_loc.transAxes)
        ax_loc.set_title("Pitch Locations")

    # Movement plot (fixed ±25 in)
    if SHOW_MOVEMENT and HB and IVB:
        ax_mv = fig.add_subplot(gs[5:7,4:6]); pretty_ax(ax_mv)
        plotted_any = False
        for tname in types:
            dd = sub.loc[sub[PTYPE]==tname, [HB, IVB]].copy()
            dd[HB]  = pd.to_numeric(dd[HB], errors="coerce")
            dd[IVB] = pd.to_numeric(dd[IVB], errors="coerce")
            dd = dd.dropna()
            if len(dd):
                plotted_any = True
                ax_mv.scatter(dd[HB], dd[IVB], s=20, alpha=0.55, label=tname,
                              c=COLOR_MAP.get(tname,"#555555"))
        ax_mv.axhline(0, lw=0.9, color="#999"); ax_mv.axvline(0, lw=0.9, color="#999")
        ax_mv.set_xlim(-MOV_LIMIT, MOV_LIMIT); ax_mv.set_ylim(-MOV_LIMIT, MOV_LIMIT)
        ax_mv.set_aspect("equal"); ax_mv.set_title(f"Pitch Movement (±{MOV_LIMIT} in)")
        ax_mv.set_xlabel("Horizontal Break (in)"); ax_mv.set_ylabel("Induced Vertical Break (in)")
        if plotted_any:
            ax_mv.legend(fontsize=7, frameon=False)
        else:
            ax_mv.text(0.5, 0.5, "No HB/IVB data", ha="center", va="center",
                       transform=ax_mv.transAxes)

    fig.tight_layout()
    return fig

# --- EXPORT LOOP ---
pitchers = [p for p in sorted(df_all[PITCHER].dropna().unique()) if str(p).strip()!=""]
print(f"Found {len(pitchers)} pitchers.")

pdf_path = os.path.join(OUT_DIR,"All_Pitchers_Reports.pdf")
with PdfPages(pdf_path) as pdf:
    for p in pitchers:
        sub = df_all[df_all[PITCHER]==p].copy()
        fig = draw_pitcher_page(sub,str(p))
        png_path = os.path.join(OUT_DIR,f"{str(p).replace('/','-')}_report.png")
        fig.savefig(png_path,bbox_inches="tight")
        pdf.savefig(fig,bbox_inches="tight")
        plt.close(fig)

print("Per-pitcher PNGs →", OUT_DIR)
print("Combined PDF    →", pdf_path)


Found 4 pitchers.


/tmp/ipython-input-3889370399.py:52: RuntimeWarning: Mean of empty slice
  def avg(s): s = pd.to_numeric(s,errors="coerce"); return round(np.nanmean(s),1) if len(s) else np.nan
/tmp/ipython-input-3889370399.py:52: RuntimeWarning: Mean of empty slice
  def avg(s): s = pd.to_numeric(s,errors="coerce"); return round(np.nanmean(s),1) if len(s) else np.nan


Per-pitcher PNGs → /content/pitcher_reports
Combined PDF    → /content/pitcher_reports/All_Pitchers_Reports.pdf


In [ ]:
from google.colab import files
files.download("/content/pitcher_reports/All_Pitchers_Reports.pdf")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip -q install nbconvert

from pathlib import Path
from google.colab import files

# Find notebooks in Colab
notebooks = list(Path("/content").glob("*.ipynb"))

print("Available notebooks:")
for nb in notebooks:
    print(nb)

Available notebooks:
/content/Pitchers_Report.ipynb


In [ ]:
from google.colab import files

uploaded = files.upload()
notebook_path = next(iter(uploaded))

print(f"Notebook found: {notebook_path}")

Saving Pitchers_Report.ipynb to Pitchers_Report.ipynb
Notebook found: Pitchers_Report.ipynb


In [ ]:
from pathlib import Path
from IPython.display import display
from google.colab import files

# Select your notebook
notebook_path = Path("/content/Pitchers_Report.ipynb")

# Export to HTML
!jupyter nbconvert --to html "{notebook_path}" \
    --output "Notebook_Report.html" \
    --output-dir "/content"

# Download HTML
files.download("/content/Notebook_Report.html")

[NbConvertApp] Converting notebook /content/Pitchers_Report.ipynb to html
[NbConvertApp] Writing 388102 bytes to /content/Notebook_Report.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>